# Defensive scraping

Peter Ralph  
2026-02-13

# Being certain about *Clueless*

We’ve been working on scraping data. Exploratory analysis in general
tends to be informal, seat-of-the-pants. This means there’s always a
tension between efficiency (so you can focus on the big picture without
getting distracted by details) and making sure you’re doing things
correctly.

Regular well-placed *tests* provide a simple defense against error. For
instance, you might check that the number of entries and number of
missing values has not changed (except when you expect it to have
changed) with something like:

    assert df.shape[0] == num_data_points
    assert df.isna().sum() == num_missing_values

Today, we’ll continue working on scraping
[Clueless](https://imsdb.com/scripts/Clueless.html), and implement some
more rigorous tests along the way.

## The basic set-up:

As in class, we’ll use [Beautiful
Soup](https://www.crummy.com/software/BeautifulSoup/bs4/doc/) to parse
the HTML.

In [1]:
import requests
import bs4
import re
import pandas as pd

session = requests.Session()
html_string = session.get("https://imsdb.com/scripts/Clueless.html")

Below is a first, imperfect, pass at parsing the document.

Your job is to write tests that check if everything “looks good”, in the
following ways:

1.  Pull all the lines that correspond to character names (e.g.,
    `"TAI\n"`) out of `all_text` using a regular expression. The number
    of times each character appears in `all_text` should be equal
    (perhaps with understood exceptions?) to the number of lines the
    characters have. If this is not true, then fix the parsing code so
    it does. (*Note:* “CHER V.O.” is the same character as “CHER”.)

2.  Every line in `all_text` (except the header) should appear somewhere
    in the result, either as character names, as description, or in
    `lines`. Write code to check this, and print out missing bits. Fix
    the parsing code so it gets everything. Check your fix didn’t break
    the stuff in (1).

3.  Directions are enclosed in parentheses. Check that these always
    appear in the `direction` column of `info`, and not in the `lines`
    object. Fix, if not.

*Note:* this HTML is *not* totally “compliant”, and so different parsers
give different document trees! Try others (`"html5lib"` or `"lxml"` in
place of `"html.parser"`) to see if they give more consistent results.

## Parsing first pass

Here’s parsing: (*note:* install `html5lib` for this to work)

In [29]:
doc = bs4.BeautifulSoup(html_string.content, 'html5lib')

def parse(doc):
    dir_re = re.compile("([(][^)]*[)])")
    lines = []
    info = []
    who = None
    start = False
    for t in doc.pre.find_all("font", size=True):
        if (not start) and re.search("SCENE I", t.text):
            start = True
        if not start:
            continue
        s = t.attrs['size']
        if s == "+1":
            if t.parent.parent.parent.name == "blockquote":
                what = 'dialog'
                info.append((what, who, direction))
                lines.append(t.text.replace("\n", " "))
            elif t.parent.has_attr("color"):
                what = "direction"
                info.append((what, "", t.text.replace("\n", " ")))
                lines.append("")
            else:
                who = t.text
                direction = ""
        else:
            what = "scene"
            info.append((what, "", t.text.replace("\n", " ")))
            lines.append("")
    info = pd.DataFrame.from_records(info, columns=["what", "who", "direction"])
    return info, lines

info, lines = parse(doc)`

We can get the entire script as follows:

In [32]:
all_text = doc.pre.text.split("\n")[112:]

In [31]:
for l in [l for l in lines if len(l.strip()) > 0][:10]:
    print(l)

So OK, you're probably thinking, "Is this, like a Noxema commercial, or what?!" But seriously, I actually have a way normal life for a teenage girl. I mean I get up, I brush my teeth, and I pick out my school clothes.
Daddy's a litigator. Those are the scariest kinds of lawyers. Even Lucy, our maid, is terrified of him. He's so good he gets paid five hundred dollars an hour just to fight with people, but he fights with me for free 'cause I'm his daughter.
Daddy!
Cher, please don't start with the juice again.
Daddy, you need your vitamin C.
Where's my briefcase?
It's been a couple of months now, so I say we go out to Malibu.
Don't tell me those braindead low-lifes have been calling again.
They are your parents. And don't try sneaking out of the office. Dr. Lovitz is coming by to give you a flu shot.
Oh, Josh is in town. He's coming for dinner.


In [16]:
for t in doc.pre.find_all("font", size=True)[20:26]:
    print(t.text)

Where's my briefcase?
CHER
It's been a couple of months now, so I say we go out
to Malibu.
MEL
Don't tell me those braindead low-lifes have been
calling again.
CHER


## Counting characters

Here's a go at part (a).

In [54]:
import collections
# regular expression is:
#  "start of line, then some spaces,
#  then a string of at least three capital letters intermixed with some punctuation and spaces"
reg = re.compile("^ *[A-Z,.:' ]{3,}")
matches = [re.search(reg, t) is not None for t in all_text]
match_lines = [t for z, t in zip(matches, all_text) if z]
char_lines = [t for t in match_lines if t[:5] != "SCENE"]
char_lines = [
    re.sub(" *[(].*", "", t).strip()
    for t in char_lines
    if t[:2] != "OK"
]
# remove lines with ?, !, #
# and also "THE END" or "END CREDITS" or a few more things
# and while we're about it, remove the "V.O."s (for "voice over")
char_lines = [
    t.replace(" V.O.", "")
    for t in char_lines
    if re.search("[!?#]", t) is None
        and not (t in  ("THE END", "END CREDITS", "DISASTER", "DINING ROOM", "HOUSE"))
]
char_counts = collections.Counter(char_lines)
char_counts

Counter({'CHER': 380,
         'TAI': 95,
         'DIONNE': 86,
         'JOSH': 83,
         'MEL': 74,
         'CHRISTIAN': 43,
         'TRAVIS': 41,
         'MURRAY': 37,
         'ELTON': 30,
         'MR HALL': 17,
         'MISS GIEST': 10,
         'LAWYER': 10,
         'AMBER': 9,
         'MISS STOEGER': 7,
         'SUMMER': 5,
         'LAWRENCE': 3,
         'OPERATOR': 3,
         'ROBBER': 3,
         'GAIL': 3,
         'STUDENT': 3,
         'CHER & DIONNE': 2,
         'HEATHER': 2,
         'CLASSMATES': 1,
         'PRINCIPAL': 1,
         'COLLEGE GUY': 1})

Looks like we're missing some characters! Notably, `TAI`, who has 95 lines?

In [59]:
for c in set(char_lines):
    if c not in info['who'].values:
        print(f"missing from 'who': {c}")

missing from 'who': GAIL
missing from 'who': LAWYER
missing from 'who': STUDENT
missing from 'who': LAWRENCE
missing from 'who': OPERATOR
missing from 'who': SUMMER
missing from 'who': PRINCIPAL
missing from 'who': COLLEGE GUY
missing from 'who': TAI
missing from 'who': ROBBER


Also, we've got some entries in `info['who']` that aren't characters!
For instance, we've got both `"CHER"` and `"CHER "`;
and we haven't removed the directions in a few places.

In [62]:
for c in set(info['who'].values):
    if c not in char_lines:
        print(f"found in 'who' but not char_lines: '{c.replace("\n", " ")}'")

found in 'who' but not char_lines: ''
found in 'who' but not char_lines: 'CHER '
found in 'who' but not char_lines: 'CHER V.O. '
found in 'who' but not char_lines: 'JOSH '
found in 'who' but not char_lines: 'CHER (from upstairs)'
found in 'who' but not char_lines: 'MEL '
found in 'who' but not char_lines: 'JOSH (grabs Cher's tummy)'
found in 'who' but not char_lines: 'JOSH (to Cher)'
found in 'who' but not char_lines: 'MEL (From Dining Room)'
found in 'who' but not char_lines: 'TRAVIS '
found in 'who' but not char_lines: 'MEL (to Cher)'
found in 'who' but not char_lines: 'CHER V.O.'
found in 'who' but not char_lines: 'CHER (on phone)'
found in 'who' but not char_lines: 'MEL (in background)'
found in 'who' but not char_lines: '(Tai sits up and hits her head on the light. What a clutz!)'
found in 'who' but not char_lines: 'JOSH (to Mel)'
found in 'who' but not char_lines: '(Josh and Cher walk back to the lounge where Tai is watching T.V. and singing along with the "Mentos" ad. God I hate

Let's see how well the number of lines for each agrees?
Hm, not so well: CHER has 380 lines but we're only getting 94 of them.

In [76]:
info_counts = info['who'].value_counts()
for c in char_counts:
    if c in info['who'].values:
        nc = info_counts[c]
    else:
        nc = 0
    print(f"{c} in info: {nc}; in lines: {char_counts[c]}")

CHER in info: 94; in lines: 380
MEL in info: 33; in lines: 74
DIONNE in info: 37; in lines: 86
MURRAY in info: 6; in lines: 37
MR HALL in info: 19; in lines: 17
AMBER in info: 2; in lines: 9
ELTON in info: 1; in lines: 30
TRAVIS in info: 5; in lines: 41
JOSH in info: 26; in lines: 83
MISS STOEGER in info: 1; in lines: 7
MISS GIEST in info: 3; in lines: 10
CLASSMATES in info: 1; in lines: 1
CHER & DIONNE in info: 1; in lines: 2
PRINCIPAL in info: 0; in lines: 1
TAI in info: 0; in lines: 95
SUMMER in info: 0; in lines: 5
LAWRENCE in info: 0; in lines: 3
OPERATOR in info: 0; in lines: 3
ROBBER in info: 0; in lines: 3
HEATHER in info: 1; in lines: 2
CHRISTIAN in info: 10; in lines: 43
COLLEGE GUY in info: 0; in lines: 1
GAIL in info: 0; in lines: 3
STUDENT in info: 0; in lines: 3
LAWYER in info: 0; in lines: 10


Where's the missing things?
Looking at the script, one of TAI's lines is `"Oh, thank you."`.
That doesn't appear in any of the `lines`:

In [82]:
[t for t in lines if t.find("Oh, thank you.") >= 0]

[]

It *does* appear in `info`, under `direction`, though:

In [88]:
import numpy as np
k = np.where(info['direction'].str.find("Oh, thank you.") >= 0)[0]
info.iloc[k,:]

,what,who,direction
358,direction,,"Oh, thank you."


What happened to "TAI"? Let's look at the surrounding lines.
Ah ha; all this is being classified as "direction". 

In [94]:
for k in range(357, 361):
    print("-----", k)
    print(info.iloc[k,:])
    print(lines[k])

----- 357
what         direction
who                   
direction          TAI
Name: 357, dtype: str

----- 358
what              direction
who                        
direction    Oh, thank you.
Name: 358, dtype: str

----- 359
what         direction
who                   
direction         CHER
Name: 359, dtype: str

----- 360
what                           direction
who                                     
direction    How do you like California?
Name: 360, dtype: str



So, what's causing the parsing code to fail?
Looking at the HTML with the inspector, this bit of text is enclosed in `<font color='#000000'>...</font>` tags.
This is leading to **two** problems:

**First:** we're identifying dialog like this:
```
            if t.parent.parent.parent.name == "blockquote":
```
and this extra `font` tag is meaning we'd need `t.parent.parent.parent.parent.name` instead.
Let's replace this with just "is this text within a blockquote at *some* level:
```
            if t.find_parent("blockquote") is not None:
```
Here's the updated parsing code, which I'm copy-paste-editing below for clarity:

**Second:** we're identifying direction like this:
```
            elif t.parent.has_attr("color"):
```
since direction always is colored red or blue or something; but we didn't count on "other stuff is colored black".
Referring to [the docs](https://www.crummy.com/software/BeautifulSoup/bs4/doc/#find-all),
we
So, let's change that to
```
            elif t.parent.has_attr("color") and t.parent.get_attr("color") != "#000000":
```


In [105]:
# how's this work? ah okay
t = doc.find("font", color="#000000")
t.attrs['color']

'#000000'

In [106]:
def parse(doc):
    dir_re = re.compile("([(][^)]*[)])")
    lines = []
    info = []
    who = None
    start = False
    for t in doc.pre.find_all("font", size=True):
        if (not start) and re.search("SCENE I", t.text):
            start = True
        if not start:
            continue
        s = t.attrs['size']
        if s == "+1":
            if t.find_parent("blockquote") is not None:
                what = 'dialog'
                info.append((what, who, direction))
                lines.append(t.text.replace("\n", " "))
            elif t.parent.has_attr("color") and t.parent.attrs['color'] != "#000000":
                what = "direction"
                info.append((what, "", t.text.replace("\n", " ")))
                lines.append("")
            else:
                who = t.text
                direction = ""
        else:
            what = "scene"
            info.append((what, "", t.text.replace("\n", " ")))
            lines.append("")
    info = pd.DataFrame.from_records(info, columns=["what", "who", "direction"])
    return info, lines

info, lines = parse(doc)

Hey, well this is promising! We have all the characters now!

In [108]:
print("Missing from 'who':")
print("---------------")
for c in set(char_lines):
    if c not in info['who'].values:
        print(f"missing from 'who': {c}")
print("---------------")

Missing from 'who':
---------------
---------------


We've also got a lot more of the lines!
Hm, we even have a few *extra* lines for some of the characters.

In [109]:
info_counts = info['who'].value_counts()
for c in char_counts:
    if c in info['who'].values:
        nc = info_counts[c]
    else:
        nc = 0
    print(f"{c} in info: {nc}; in lines: {char_counts[c]}")

CHER in info: 351; in lines: 380
MEL in info: 67; in lines: 74
DIONNE in info: 85; in lines: 86
MURRAY in info: 34; in lines: 37
MR HALL in info: 19; in lines: 17
AMBER in info: 9; in lines: 9
ELTON in info: 30; in lines: 30
TRAVIS in info: 38; in lines: 41
JOSH in info: 81; in lines: 83
MISS STOEGER in info: 6; in lines: 7
MISS GIEST in info: 11; in lines: 10
CLASSMATES in info: 1; in lines: 1
CHER & DIONNE in info: 2; in lines: 2
PRINCIPAL in info: 1; in lines: 1
TAI in info: 94; in lines: 95
SUMMER in info: 5; in lines: 5
LAWRENCE in info: 3; in lines: 3
OPERATOR in info: 3; in lines: 3
ROBBER in info: 6; in lines: 3
HEATHER in info: 2; in lines: 2
CHRISTIAN in info: 46; in lines: 43
COLLEGE GUY in info: 1; in lines: 1
GAIL in info: 3; in lines: 3
STUDENT in info: 3; in lines: 3
LAWYER in info: 10; in lines: 10


For instance, we counted 6 lines for ROBBER in `info` but only 3 by our other method. What's the difference?

In [110]:
for l, c in zip(lines, info['who']):
    if c == "ROBBER":
        print(l)

Hand it over.
Give me the phone.
OK. Bag, too. C'mon! Alright, now, uh, get down on the ground. Face down. C'mon!
An a-what-a?
And I will totally shoot you in the head. Get down!
Alright, um, count to a hundred. Thank you.


Hm, all those look okay. And, looking at the script, there's things like this:
```
ROBBER

    Hand it over.

(Cher squeals/moans)

    Give me the phone.
```
This is correctly counted as two lines for the ROBBER,
whereas `ROBBER` only appears once.
So perhaps the discrepancy we're seeing there is okay!

Our next step would be to iterate: actually find some of the missing lines, and investigate where those went to.